<a href="https://colab.research.google.com/github/NizarDag/llm-post-training-labs/blob/main/01_base_vs_sft_vs_rl_gsm8k/01_base_vs_sft_vs_rl_gsm8k.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Comparing Base, SFT, and RL-Tuned Models on GSM8K

## Objective

Compare how post-training changes model behavior on mathematical reasoning tasks.

## Concepts

- Pre-training
- Supervised Fine-Tuning (SFT)
- Reinforcement Learning (RL)
- Reasoning
- GSM8K

## Dataset

GSM8K

## Models

- Base model
- SFT model
- RL model

## Results

Work in progress.

## Experiment Setup

### Research Question

How does post-training affect mathematical reasoning performance?

Specifically, we compare:

1. A Base Model (pre-trained only)
2. A Supervised Fine-Tuned (SFT) Model
3. A Reinforcement Learning (RL) Model

on GSM8K mathematical reasoning tasks.

### Expected Outcomes

- The Base Model may struggle with instruction following and reasoning.
- The SFT Model should follow instructions more reliably.
- The RL Model should demonstrate stronger reasoning behavior and higher accuracy.

In [1]:
!pip install -q datasets transformers accelerate huggingface_hub bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.5 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset

gsm8k_dataset = load_dataset("openai/gsm8k", "main")
gsm8k_dataset

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})

In [3]:
example = gsm8k_dataset["train"][0]
print("Question:")
print(example["question"])

print("\nAnswer:")
print(example["answer"])

Question:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Answer:
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72


## Dataset Exploration

GSM8K (Grade School Math 8k) contains mathmatical work promblems and detailed reasoning solutions.

Each example contains:

- question
- answer

The answer filed contains both the reasoning process and the final answer.

In [4]:
import re

def extract_gsm8k_answer(answer_text):
  match = re.search(r"####\s*(-?\d+\.?\d*)", answer_text)

  if match:
    return match.group(1)

  return None

In [5]:
golden_answer = extract_gsm8k_answer(example["answer"])
print(golden_answer)

72


In [6]:
for i in range(3):
  print("="* 80)
  print("Question:")
  print(gsm8k_dataset['test'][i]['question'])

  print("\nFinal Answer:")
  print(extract_gsm8k_answer(gsm8k_dataset['test'][i]['answer']))

Question:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Final Answer:
18
Question:
A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?

Final Answer:
3
Question:
Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?

Final Answer:
70000


In [7]:
!nvidia-smi

Thu Jun 18 17:09:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
model_names = {
    "base": "deepseek-ai/deepseek-math-7b-base",
    "sft": "deepseek-ai/deepseek-math-7b-instruct",
    "rl": "deepseek-ai/deepseek-math-7b-rl"
}

In [9]:
from transformers import AutoTokenizer

In [10]:
for name, model_id in model_names.items():
  print(f"testing {name}: {model_id}")

  try:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    print("Found")

  except Exception as e:
    print("Failed")
    print(e)

    print("-" * 50)

testing base: deepseek-ai/deepseek-math-7b-base


config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.61M [00:00<?, ?B/s]

Found
testing sft: deepseek-ai/deepseek-math-7b-instruct


...

Found
testing rl: deepseek-ai/deepseek-math-7b-rl


config.json:   0%|          | 0.00/626 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.61M [00:00<?, ?B/s]

Found


## Model Verification

The follwoing models were successfully located on Hugging Face:
- deepseek-ai/deepseek-math-7b-base
- deepseek-ai/deepseek-math-7b-instruct
- deepseek-ai/deepseek-math-7b-rl

These represent three stages of the LLM training pipline:

1. Base Model (pretraining)
2. Supervised Fine-tuned Model (SFT)
3. Reinforcement Learning Model (RL)

In [2]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_use_double_quant=True
)

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model(model_name):
  tokenizer = AutoTokenizer.from_pretrained(model_name, turst_remote_code = True)
  model = AutoModelForCausalLM.from_pretrained(
      model_name,
      trust_remote_code=True,
      quantization_config = bnb_config,
      device_map = "auto"
  )

  return tokenizer, model

In [13]:
def generate_answer(tokenizer, model, question):

    prompt = f"""
Solve the following math problem.

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    print("Input device:", inputs["input_ids"].device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=128,
            do_sample=False
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response

In [5]:
base_tokenizer, base_model = load_model(
    model_name=model_names["base"])

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [6]:
test = base_tokenizer(
    "hello world",
    return_tensors="pt"
)

for k, v in test.items():
    print(k, type(v))

input_ids <class 'torch.Tensor'>
attention_mask <class 'torch.Tensor'>


In [7]:
def generate_answer(tokenizer, model, question):

    prompt = f"""
Solve the following math problem.

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    print("DEBUG")
    for k, v in inputs.items():
        print(k, type(v))

    return inputs

In [8]:
question = gsm8k_dataset["test"][0]["question"]

generate_answer(
    base_tokenizer,
    base_model,
    question
)

DEBUG
input_ids <class 'torch.Tensor'>
attention_mask <class 'torch.Tensor'>


{'input_ids': tensor([[19972, 16370,   247, 26598,   664, 25114,    13, 23853,    25, 14957,
          1550,   678,  6601,  3582,    16,    21,   613,  4817,   524,  1356,
            13,  4622, 36134, 55672,   758,  1467,  9343,    69,  5102,   644,
         29047,   384,    65,  1778,  1946,   539,  1233,  1467,   397,  8149,
           346,   644,  1356,  2296, 14741,    13,    50,  1218,   488,   292,
          1535,  5622,   583,   253,   672, 49552,   408,     6, 25649, 41882,
          1467,     3,    17,   524, 40555,  7621,   400,  1817,    13,  2819,
         17031,   515, 48930,  2860,   390,   247,  7690, 11576,  1356,   253,
           672, 49552,   408,     6, 25649,    30, 32349,    25]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1

## Behavioral Evaluation

Rather than evaluating only correctness, this notebook also compare:

- Instruction Following
- Reasoning Style
- Output Format
- Mathematical Accuracy

to understand how post-training changes model behavior.

In [14]:
question = gsm8k_dataset["test"][0]["question"]

response = generate_answer(
    base_tokenizer,
    base_model,
    question
)

print(response)

[transformers] Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.


Input device: cuda:0


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Ċ-Ġ1.ĠSolveĠtheĠfollowingĠmathĠproblem.ĠQuestion:ĠJanetĠducksĠlayĠ16ĠeggsĠperĠday.ĠSheĠeatsĠthreeĠforĠbreakfastĠeveryĠmorningĠandĠbakesĠmuffinsĠforĠherĠfriendsĠeveryĠdayĠwithĠfour.ĠSheĠsellsĠtheĠremainderĠatĠtheĠfarmers'ĠmarketĠdailyĠforĠ$2ĠperĠfreshĠduckĠegg.ĠHowĠmuchĠinĠdollarsĠdoesĠsheĠmakeĠeveryĠdayĠatĠtheĠfarmers'Ġmarket?ĠAnswer:Ċ-Ġ2.ĠSolveĠtheĠfollowingĠmathĠproblem.ĠQuestion:ĠJanetĠducksĠlayĠ16ĠeggsĠperĠday.ĠSheĠeatsĠthreeĠforĠbreakfastĠeveryĠmorningĠandĠbakesĠmuffinsĠforĠherĠfriendsĠeveryĠdayĠwithĠfour.ĠSheĠsellsĠtheĠremainderĠatĠtheĠfarmers


In [28]:
test = base_tokenizer(
    "Hello world",
    return_tensors="pt"
)

print(test)

{'input_ids': tensor([[  39, 3857, 3982]]), 'attention_mask': tensor([[1, 1, 1]])}


In [33]:
question = gsm8k_dataset["test"][0]["question"]

test_inputs = base_tokenizer(
    question,
    return_tensors="pt"
)

print(type(test_inputs))

for k, v in test_inputs.items():
    print(k, type(v))

<class 'transformers.tokenization_utils_base.BatchEncoding'>
input_ids <class 'torch.Tensor'>
attention_mask <class 'torch.Tensor'>
